# 航班价格监控 · EDA 01 — 价格轨迹与购买时机

核心问题：
1. **提前多少天买通常更便宜？**（把各轨迹按“距起飞天数 lead”对齐）
2. **等一等 vs 现在买，胜率是多少？**
3. **“今天该不该买”怎么看**（现价在同期历史里的分位）
4. **去程 / 回程（北京⇄泉州）价格是否对称？**

口径提示（重要）：
- `lead = 起飞日 - 爬取日`，lead 大 = 很早看，lead 小 = 临近起飞。
- **删失（censored）**：某航班在某起飞日临近前就从抓取结果消失，极可能已售罄；
  这种轨迹在 grid 里**不会出现临期价**，所以“临期便宜/贵”只统计到真实可订的时刻。
- **滚动窗口**：7 月那批起飞日只能从很短的 lead 观察；lead≥45 天的观察只来自 8–10 月的起飞日，
  两者需求季节不同，跨 lead 比“绝对值”时会被季节混淆 —— 因此重点看**同一航班内部**的涨跌，
  以及**当日全网最低**的走势。

## 0. 准备（含较耗时的轨迹网格化，约 10–20 秒）

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "eda"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import common as C

mpl.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
                     "axes.grid": True, "axes.axisbelow": True})
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
sns.set_palette(PALETTE)

df = C.load_flight_prices()
df = C.add_features(df)
grid = C.lead_grid(df)                     # 每条轨迹按整数 lead 展开
floor = C.floor_by_lead(grid)              # (航向,起飞日,lead桶) 当日全网最低
trans = C.lead_transitions(grid)           # 轨迹在各目标 lead 上的价
print("grid 行数:", len(grid), "| floor 行数:", len(floor), "| transitions 行数:", len(trans))

DIRS = ["北京→泉州", "泉州→北京"]          # 主线往返两方向
DIR_COLOR = {"北京→泉州": PALETTE[0], "泉州→北京": PALETTE[1]}

## 1. 当日全网最低价，随“距起飞时间”的典型走势

对某起飞日，在“距起飞还有 X 天”那一刻，全网能订到的最低价（`floor`）是多少。
把所有起飞日横向拼起来，按 lead 分桶看中位数与四分位——**越接近起飞，通常越贵还是越便宜？**
每桶右侧标出贡献的“起飞日×航向”组合数 n（样本少时结论要打折）。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, rl in zip(axes, DIRS):
    sub = floor[floor["route_label"] == rl]
    stats = (sub.groupby("lead_bin", observed=True)["floor"]
             .agg(n="size", lo=lambda x: x.quantile(.25), med="median",
                  hi=lambda x: x.quantile(.75), avg="mean"))
    stats = stats.reindex([b for b in C.LEAD_BIN_ORDER if b in stats.index])
    x = np.arange(len(stats))
    ax.plot(x, stats["med"], "-o", color=DIR_COLOR[rl], lw=2, ms=5,
            label="中位数 floor")
    ax.fill_between(x, stats["lo"], stats["hi"], color=DIR_COLOR[rl], alpha=0.18,
                    label="25–75 分位")
    ax.set_xticks(x); ax.set_xticklabels(stats.index, rotation=45, ha="right")
    ax.set_title(f"{rl}   当日全网最低的走势")
    ax.set_ylabel("价格(元)")
    for xi, n in zip(x, stats["n"]):
        ax.annotate(f"n={int(n)}", (xi, stats["med"].max() * 1.03), ha="center",
                    fontsize=7, color="#898781")
    ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

1.结论：两方向的“当日全网最低”在 **45 天～6 天窗口内都保持约 420/400 元的平台**（中位波动 <50 元，见 n 标注的桶样本量），**1–3 天升至 480/450 元，起飞当天跳升到 870/680 元**（去程 25–75 分位 350–1030，区间明显变宽）。即“早买早便宜”并不成立——价格在前两周几乎是平的，真正的风险在**临期 1–2 天的跳涨**；且起飞当天只剩“幸存者”（便宜舱已删失，见 02 §6），该读数应理解为“还能买到的下限”。

> 读图：若曲线向左（临近起飞）整体抬升，说明“越等越贵”，早买占优；
> 若在临期段反而下探，说明存在未售临期甩卖（注意此时很多便宜航班已删失/售罄，观察的是“幸存者”）。
> 两个方向曲线形状可以对照（去程 9/30–10/1 国庆窗口的贡献也会体现在‘1-3天’这类桶里）。

## 2. “等一等 vs 现在买”：同一条轨迹内，不同 lead 窗口的涨跌胜率

对每一条轨迹，比较“提前 A 天”与“更临近的提前 B 天”（B<A）两个时点看到的价。
因为比的是**同一个航班同一个起飞日**，消除了航班/航司/季节的基准价差异。

表格读法：从提前 A 天**等到**提前 B 天再买——
  - “跌%” = 价格下降、等到了更便宜（等待占优）
  - “涨%” = 反而更贵
  - 中位Δ = 中位价差（负 = 平均省了这么多）

In [ ]:
piv = trans.pivot_table(index=["traj_key", "route_label", "flight_date"],
                        columns="target_lead", values="price")
PAIRS = [(45, 30), (30, 21), (30, 14), (21, 14), (21, 7), (14, 7), (14, 3), (7, 3), (7, 1)]
tab = []
for rl in DIRS:
    sub = piv[piv.index.get_level_values("route_label") == rl]
    for a, b in PAIRS:
        if a not in sub.columns or b not in sub.columns:
            continue
        ok = sub[[a, b]].dropna()
        if len(ok) < 20:
            continue
        d = ok[b] - ok[a]
        tab.append({"方向": rl, "等待窗口": f"{a}→{b}天",
                    "轨迹数": len(ok),
                    "跌%": round((d < 0).mean() * 100),
                    "平%": round((d == 0).mean() * 100),
                    "涨%": round((d > 0).mean() * 100),
                    "中位Δ(元)": round(d.median()),
                    "均值Δ(元)": round(d.mean())})
tt = pd.DataFrame(tab)
tt

### 2.1 用条形可视化“跌/涨”胜率（去程 + 回程）
每根条代表一个“等待窗口”，拆成跌/平/涨三段——直观看出哪个窗口等下去是赚多赔少。

In [ ]:
import matplotlib.colors as mcol
SEG = {"跌": PALETTE[2], "平": "#b9b7ad", "涨": PALETTE[7]}

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for ax, rl in zip(axes, DIRS):
    t = tt[tt["方向"] == rl].copy().sort_values("等待窗口", key=lambda s: s.map({
        f"{a}→{b}天": a - b for a, b in PAIRS}))
    bottom = np.zeros(len(t))
    for k, col in [("跌%", "跌"), ("平%", "平"), ("涨%", "涨")]:
        vals = t[k].to_numpy()
        ax.bar(t["等待窗口"], vals, bottom=bottom, label=col,
               color=SEG[col], width=.62, edgecolor="white", linewidth=.6)
        bottom += vals
    ax.set_title(f"{rl}  从提前 A 天等到提前 B 天，价格变化方向占比")
    ax.set_ylim(0, 100); ax.set_ylabel("占比 %")
    ax.axhline(50, color="#898781", lw=.6, ls="--")
    ax.tick_params(axis="x", rotation=45)
    if ax == axes[0]:
        ax.legend(fontsize=8, loc="upper right")
axes[0].set_xlabel("等待窗口（A→B 天）")
fig.tight_layout(); plt.show()

2.结论：条形图与上表一致——**等待 1–2 周是占优策略**：30→21、30→14 天窗口“跌”占 51–53%、中位价差 **−20～−30 元**，且平局占比高（18–34%）；21→14、14→7 是中性窗口（涨跌各约 1/3）；**一周内再等明显吃亏：7→1 天窗口跌仅 24–28%、涨 62–66%、中位 +90～+170 元**。两方向结论一致——买点节奏是“提前 2–4 周锁定，1 周内尽快决策”。

## 3. “今天该不该买”——现价在同期历史里的分位

取**最近一次爬取批次**为“今天”，把每个仍可订的未来起飞日（距起飞 1–30 天）算一个
“当日全网最低价 floor_now”，再和**历史同期**（同一航向、同一 lead 分桶）的 floor 分布比，
看现在处在该分布的什么分位：
  - 分位低（≤25%）→ 已经是同期里便宜的位置，可下手；
  - 分位高（≥75%）→ 明显高于同期常见价，观望更稳。

下表按起飞日从近到远列 12 个最近仍可订的日期。

In [ ]:
hist = (floor.groupby(["route_label", "lead_bin"])["floor"]
        .quantile([.25, .5, .75]).unstack()
        .rename(columns={.25: "P25", .5: "P50", .75: "P75"})
        .reset_index())

last_t = df["crawl_dt"].max()
last = df[df["crawl_dt"] == last_t].copy()
last["lead_bin"] = last["lead_days"].map(C.lead_bin)
now_floor = (last.groupby(["route_label", "flight_date"])
             .agg(floor_now=("price", "min"), n_flights=("flight_no", "nunique"),
                  lead_now=("lead_days", "min"))
             .reset_index())
now_floor = now_floor[(now_floor["lead_now"] >= 1) & (now_floor["lead_now"] <= 30)]
now_floor["lead_bin"] = now_floor["lead_now"].map(C.lead_bin)

m = now_floor.merge(hist, on=["route_label", "lead_bin"], how="left")
m = m[m["P25"].notna()].copy()

def pctile_row(r):
    qs = [r["P25"], r["P50"], r["P75"]]
    if r["floor_now"] <= qs[0]:
        return "≤25%·偏便宜"
    if r["floor_now"] <= qs[1]:
        return "25–50%"
    if r["floor_now"] <= qs[2]:
        return "50–75%"
    return "≥75%·偏贵"
m["现价分位"] = m.apply(pctile_row, axis=1)
m["建议"] = np.select([m["现价分位"].str.contains("偏便宜"),
                       m["现价分位"].str.contains("偏贵")],
                      ["可考虑下手", "观望"], default="中性")

show = (m.sort_values("flight_date").head(24))
show = show.sort_values("flight_date")
show[["route_label", "flight_date", "lead_now", "floor_now", "n_flights",
      "P25", "P50", "P75", "现价分位", "建议"]].rename(
    columns={"route_label": "航向", "flight_date": "起飞日", "lead_now": "距起飞(天)",
             "floor_now": "当前全网最低", "n_flights": "可选航班数",
             "P25": "同期P25", "P50": "同期P50", "P75": "同期P75"})

3.结论：以最近批次（9/3 11:29）为“今天”：未来 1–12 天里，北京→泉州有 6 个起飞日（9/4、9/5、9/6、9/7、9/8、9/10）现价 ≤ 同期 P25，判为“可考虑下手”（如 9/4 全网最低 300 vs 同期 P25 357.5）；泉州→北京仅 9/10、9/14 两日偏便宜。**9/11–9/14 去程与 9/12 回程落在 ≥75%·偏贵**（如去程 9/14 最低 470 vs P75 440）→ 观望。注意该表只覆盖“距起飞 1–30 天”的日期，且同期基准含 7 月短观察窗口，读作“相对便宜/贵”而非绝对建议。

> 上面的“同期”是**同一航向、同一 lead 桶**的历史 floor 分布。国庆（9/30–10/1）等高峰期的
> 现价很可能落在“≥75%·偏贵” —— 那不是系统误判，而是高峰本身贵，买点窗口要提前更多。

## 4. 该出手的提前量：每（航向,起飞日）的“全网最低”通常出现在距起飞几天

对每个起飞日，找它被观察到的最低价出现时距起飞还有几天（取第一个到达接近最低价 5% 以内的时刻），
直方图看“最低价提前 N 天出现”的分布。这回答**目标出行日要提前多久开始盯**。

In [ ]:
# 直接从 grid 按 (航向,起飞日,整数lead) 求“当日全网最低”，再找该日历史最低最早出现的 lead
per_lead_min = (grid.groupby(["route_label", "flight_date", "lead_days"])["price"]
                .min().rename("floor_by_lead").reset_index())
per_lead_min["_mind"] = (per_lead_min.groupby(["route_label", "flight_date"])
                         ["floor_by_lead"].transform("min"))
per_lead_min["_keep"] = (per_lead_min["floor_by_lead"]
                         <= per_lead_min["_mind"] * 1.05)   # 接近该日历史最低价
lead_at_min = (per_lead_min[per_lead_min["_keep"]]
               .groupby(["route_label", "flight_date"])["lead_days"]
               .max().rename("lead_at_min")                  # 最早出现该价位的 lead
               .reset_index())

fig, ax = plt.subplots(figsize=(8, 3.6))
lam = lead_at_min[lead_at_min["route_label"].isin(DIRS)]
sns.histplot(data=lam, x="lead_at_min", hue="route_label", multiple="dodge",
             palette=DIR_COLOR, discrete=True, ax=ax)
ax.axvline(7, color="#898781", ls="--", lw=.8)
ax.text(7.4, ax.get_ylim()[1] * .95, "≈提前 1 周", fontsize=8, color="#52514e")
ax.set_xlabel("该起飞日的全网最低价最早出现的 lead（距起飞天数）")
ax.set_title("全网最低价通常提前几天就能看到")
ax.legend(title=None)
fig.tight_layout(); plt.show()

lam.groupby("route_label")["lead_at_min"].describe(
    percentiles=[.25, .5, .75]).round(0)

4.结论：每个起飞日的“全网最低价”最早出现的 lead 中位 **20–21 天**（均值 17–18 天），分布呈双峰：**主峰 27–28 天**（提前近 4 周就能看到最低价）+ **次峰 2–5 天**（临期尾舱低价，量少）；少数 36–38 天是观察窗上限。→ 目标出行日应**提前 3–4 周开始每天盯**，而不是赌临期甩卖（临期低价只是少数幸存者，且很快消失，见 02 §6）。

## 5. 往返不对称：北京→泉州 vs 泉州→北京

数据里去程（29430 条）比回程（25485 条）多 —— 时刻表/航班数不同造成的。
下面从两层看不对称：(a) 全程价格水平分布；(b) 同一起飞日两方向“历史最低”的关系。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# (a) 快照价格分布
for ax, rl in zip(axes, DIRS):
    sub = df[(df["route_label"] == rl) & (df["lead_days"] >= 0)]
    sns.histplot(sub["price"], bins=45, color=DIR_COLOR[rl], ax=ax, alpha=.75)
    ax.axvline(sub["price"].median(), color="#0b0b0b", ls="--", lw=1)
    ax.text(sub["price"].median() * 1.01, ax.get_ylim()[1] * .95,
            f"中位 {sub['price'].median():.0f}", fontsize=9)
    ax.set_title(rl)
    ax.set_xlabel("价格(元)")
axes[0].set_ylabel("快照数")
fig.suptitle("(a) 全程价格快照分布（lead≥0）", y=1.02)
fig.tight_layout(); plt.show()

# (b) 同一起飞日，两方向历史最低的可比性
mn = (grid.groupby(["route_label", "flight_date"])["price"].min().rename("min")
      .reset_index())
w = mn.pivot(index="flight_date", columns="route_label", values="min").dropna(
    subset=DIRS)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(w[DIRS[0]], w[DIRS[1]], s=28, color=PALETTE[0], alpha=.75,
           edgecolor="white", lw=.5)
lim = [min(w[DIRS[0]].min(), w[DIRS[1]].min()) - 50,
       max(w[DIRS[0]].max(), w[DIRS[1]].max()) + 50]
ax.plot(lim, lim, color="#898781", ls="--", lw=.8, label="y=x")
ax.set_xlabel(f"{DIRS[0]} 历史最低"); ax.set_ylabel(f"{DIRS[1]} 历史最低")
ax.set_title("(b) 同一起飞日两方向的最低可订价")
ax.legend()
fig.tight_layout(); plt.show()

### 5.1 用分位数概括不对称
两方向同一起飞日“历史最低”之差（回程 − 去程）：正 = 回程整体更贵。

In [ ]:
diff = (w[DIRS[1]] - w[DIRS[0]]).rename("回程-去程(历史最低)")
print(diff.describe(percentiles=[.25, .5, .75]).round(0).to_string())
print("\n回程更贵的起飞日占比: %.0f%%" % (100 * (diff > 0).mean()))

fig, ax = plt.subplots(figsize=(8, 3))
sns.histplot(diff, bins=40, color=PALETTE[0], ax=ax)
ax.axvline(0, color="#0b0b0b", lw=.9)
ax.set_xlabel("回程历史最低 − 去程历史最低（元）")
ax.set_title("两方向最低价差的分布（>0 回程贵）")
fig.tight_layout(); plt.show()

5.结论：去程/回程中位价同为 600 元，但**同一起飞日的“历史最低”不对称**：回程−去程 差中位 **−40 元**、均值 −48 元，75% 的日期差在 −85～+15 元之间，仅 26% 的起飞日回程更贵；大值端（国庆 1,300–1,400 元档）回程反而比去程便宜 500+ 元（散点明显低于 y=x）。方向差异比“中位 600”暗示的大——**买票时两个方向要分开定心理价**，回程整体可预期更低。

## 小结
- lead 曲线 + 轨迹内“等待窗口”胜率给出了**买点节奏**：回看数据，
  等一两周（例如 30→14、21→14）多数时候是跌多涨少；而到了 7 天以内再等，翻转概率上升
  —— 与“低价舱在前两周放出、临期反而甩卖/售罄”两种机制都吻合，需结合你的目标日期季节性判断。
- “该不该买”应同时看**同航线同 lead 的历史分位** 和 **删失信号**（便宜航班在 1 周内快速消失）。
- 去程/回程**不对称明显且稳定**（见差分布与 y=x 散点），买票策略要分开设预期。

下一步（notebook 02）：把价格拆解成 **起飞日星期/月份 → 出发时刻 → 北京机场(大兴vs首都) → 机型 → 航司** 的定价结构，
并量化 **价格波动/告警噪音** 与 **疑似售罄**。